# Fine tuning

Fine-tune TinyLlama to create a movie-expert chatbot for CineFlow.

**Requirements:**
- Google Colab with GPU (free T4 works!)
- Hugging Face account

**Dataset:** 60+ movies, 950+ training examples

**Time:** ~30-45 minutes on T4 GPU

## 1. Setup Environment

In [2]:
%pip install -q torch transformers datasets peft trl bitsandbytes accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.0 MB/s eta 0:00:00


In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


## 2. Configuration - EDIT THIS!

In [4]:
HF_USERNAME = "gabrielniculaesei"
MODEL_NAME = "cinebot-movie-expert"

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./cinebot-output"

In [5]:
from huggingface_hub import login
login()  # Enter your HF token when prompted

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 3. Create Training Data (60+ Movies)

In [6]:
import json
import random

# 60+ popular movies across all genres
MOVIES = [
    {"title": "Inception", "year": 2010, "director": "Christopher Nolan", "genres": ["Sci-Fi", "Action"], "cast": ["Leonardo DiCaprio", "Joseph Gordon-Levitt"], "plot": "A thief who steals secrets from dreams is offered redemption by planting an idea instead.", "themes": ["dreams", "reality", "guilt"], "similar": ["The Matrix", "Interstellar", "Tenet"]},
    {"title": "The Shawshank Redemption", "year": 1994, "director": "Frank Darabont", "genres": ["Drama"], "cast": ["Tim Robbins", "Morgan Freeman"], "plot": "A banker convicted of murder befriends a fellow prisoner while maintaining hope.", "themes": ["hope", "friendship", "redemption"], "similar": ["The Green Mile", "Forrest Gump"]},
    {"title": "The Dark Knight", "year": 2008, "director": "Christopher Nolan", "genres": ["Action", "Crime"], "cast": ["Christian Bale", "Heath Ledger"], "plot": "Batman faces his greatest challenge when the Joker plunges Gotham into chaos.", "themes": ["chaos", "heroism", "sacrifice"], "similar": ["Batman Begins", "Joker", "Logan"]},
    {"title": "Pulp Fiction", "year": 1994, "director": "Quentin Tarantino", "genres": ["Crime", "Drama"], "cast": ["John Travolta", "Samuel L. Jackson"], "plot": "Interconnected stories of criminals in Los Angeles unfold non-linearly.", "themes": ["fate", "redemption", "violence"], "similar": ["Reservoir Dogs", "Kill Bill"]},
    {"title": "Interstellar", "year": 2014, "director": "Christopher Nolan", "genres": ["Sci-Fi", "Drama"], "cast": ["Matthew McConaughey", "Anne Hathaway"], "plot": "Explorers travel through a wormhole to find a new home as Earth faces extinction.", "themes": ["love", "time", "survival"], "similar": ["Gravity", "The Martian", "Arrival"]},
    {"title": "Parasite", "year": 2019, "director": "Bong Joon-ho", "genres": ["Thriller", "Drama"], "cast": ["Song Kang-ho", "Choi Woo-shik"], "plot": "A poor family schemes to infiltrate a wealthy household.", "themes": ["class inequality", "deception"], "similar": ["Snowpiercer", "Us"]},
    {"title": "The Matrix", "year": 1999, "director": "The Wachowskis", "genres": ["Sci-Fi", "Action"], "cast": ["Keanu Reeves", "Laurence Fishburne"], "plot": "A hacker discovers reality is a simulation and joins a rebellion.", "themes": ["reality", "free will"], "similar": ["Inception", "Dark City"]},
    {"title": "Everything Everywhere All at Once", "year": 2022, "director": "The Daniels", "genres": ["Sci-Fi", "Action", "Comedy"], "cast": ["Michelle Yeoh", "Ke Huy Quan"], "plot": "A woman discovers she can access parallel universe selves to save the multiverse.", "themes": ["family", "nihilism", "identity"], "similar": ["The Matrix", "Swiss Army Man"]},
    {"title": "The Godfather", "year": 1972, "director": "Francis Ford Coppola", "genres": ["Crime", "Drama"], "cast": ["Marlon Brando", "Al Pacino"], "plot": "The aging patriarch of a crime dynasty transfers control to his reluctant son.", "themes": ["family", "power", "loyalty"], "similar": ["The Godfather Part II", "Goodfellas"]},
    {"title": "Spirited Away", "year": 2001, "director": "Hayao Miyazaki", "genres": ["Animation", "Fantasy"], "cast": ["Rumi Hiiragi", "Miyu Irino"], "plot": "A young girl enters a magical world of spirits to free herself and her parents.", "themes": ["coming of age", "courage"], "similar": ["Howl's Moving Castle", "Princess Mononoke"]},
    {"title": "Whiplash", "year": 2014, "director": "Damien Chazelle", "genres": ["Drama", "Music"], "cast": ["Miles Teller", "J.K. Simmons"], "plot": "A young drummer faces an abusive instructor who pushes him to his limits.", "themes": ["obsession", "perfection"], "similar": ["La La Land", "Black Swan"]},
    {"title": "Get Out", "year": 2017, "director": "Jordan Peele", "genres": ["Horror", "Thriller"], "cast": ["Daniel Kaluuya", "Allison Williams"], "plot": "A young Black man visits his girlfriend's family and uncovers disturbing secrets.", "themes": ["racism", "identity"], "similar": ["Us", "Nope"]},
    {"title": "Mad Max: Fury Road", "year": 2015, "director": "George Miller", "genres": ["Action", "Sci-Fi"], "cast": ["Tom Hardy", "Charlize Theron"], "plot": "In a post-apocalyptic wasteland, a woman rebels against a tyrannical ruler with Max's help.", "themes": ["survival", "freedom"], "similar": ["The Road Warrior", "Dredd"]},
    {"title": "John Wick", "year": 2014, "director": "Chad Stahelski", "genres": ["Action", "Thriller"], "cast": ["Keanu Reeves", "Willem Dafoe"], "plot": "A retired hitman seeks vengeance against gangsters who killed his dog.", "themes": ["revenge", "grief"], "similar": ["Nobody", "The Equalizer"]},
    {"title": "Die Hard", "year": 1988, "director": "John McTiernan", "genres": ["Action", "Thriller"], "cast": ["Bruce Willis", "Alan Rickman"], "plot": "A cop battles terrorists in a Los Angeles skyscraper during Christmas.", "themes": ["heroism", "resilience"], "similar": ["Lethal Weapon", "Speed"]},
    {"title": "Gladiator", "year": 2000, "director": "Ridley Scott", "genres": ["Action", "Drama"], "cast": ["Russell Crowe", "Joaquin Phoenix"], "plot": "A betrayed Roman general seeks revenge against the emperor who murdered his family.", "themes": ["revenge", "honor"], "similar": ["Braveheart", "Troy"]},
    {"title": "Top Gun: Maverick", "year": 2022, "director": "Joseph Kosinski", "genres": ["Action", "Drama"], "cast": ["Tom Cruise", "Miles Teller"], "plot": "Maverick returns to train Top Gun graduates for a specialized mission.", "themes": ["legacy", "mentorship"], "similar": ["Top Gun", "Mission: Impossible"]},
    {"title": "The Shining", "year": 1980, "director": "Stanley Kubrick", "genres": ["Horror", "Drama"], "cast": ["Jack Nicholson", "Shelley Duvall"], "plot": "A family heads to an isolated hotel where a sinister presence drives the father to violence.", "themes": ["isolation", "madness"], "similar": ["Doctor Sleep", "Hereditary"]},
    {"title": "Hereditary", "year": 2018, "director": "Ari Aster", "genres": ["Horror", "Drama"], "cast": ["Toni Collette", "Alex Wolff"], "plot": "After their grandmother dies, a family unravels terrifying secrets.", "themes": ["grief", "family trauma"], "similar": ["Midsommar", "The Witch"]},
    {"title": "A Quiet Place", "year": 2018, "director": "John Krasinski", "genres": ["Horror", "Sci-Fi"], "cast": ["Emily Blunt", "John Krasinski"], "plot": "A family must live in silence to avoid creatures that hunt by sound.", "themes": ["survival", "family"], "similar": ["Bird Box", "Don't Breathe"]},
    {"title": "The Conjuring", "year": 2013, "director": "James Wan", "genres": ["Horror", "Mystery"], "cast": ["Vera Farmiga", "Patrick Wilson"], "plot": "Paranormal investigators help a family terrorized by a dark presence.", "themes": ["faith", "evil"], "similar": ["Insidious", "The Exorcist"]},
    {"title": "It", "year": 2017, "director": "Andy Muschietti", "genres": ["Horror", "Fantasy"], "cast": ["Bill Skarsgård", "Finn Wolfhard"], "plot": "Kids face an evil clown terrorizing their town.", "themes": ["friendship", "courage"], "similar": ["Stranger Things", "Stand By Me"]},
    {"title": "Superbad", "year": 2007, "director": "Greg Mottola", "genres": ["Comedy"], "cast": ["Jonah Hill", "Michael Cera"], "plot": "High school seniors try to score alcohol for a party before graduation.", "themes": ["friendship", "growing up"], "similar": ["Booksmart", "The Hangover"]},
    {"title": "The Grand Budapest Hotel", "year": 2014, "director": "Wes Anderson", "genres": ["Comedy", "Drama"], "cast": ["Ralph Fiennes", "Tony Revolori"], "plot": "A legendary concierge and his protégé become entangled in a murder mystery.", "themes": ["nostalgia", "loyalty"], "similar": ["Moonrise Kingdom", "Knives Out"]},
    {"title": "The Hangover", "year": 2009, "director": "Todd Phillips", "genres": ["Comedy"], "cast": ["Bradley Cooper", "Zach Galifianakis"], "plot": "Friends wake up from a Vegas bachelor party with no memory and the groom missing.", "themes": ["friendship", "chaos"], "similar": ["Superbad", "21 Jump Street"]},
    {"title": "Bridesmaids", "year": 2011, "director": "Paul Feig", "genres": ["Comedy", "Romance"], "cast": ["Kristen Wiig", "Melissa McCarthy"], "plot": "A woman's life unravels as she leads her best friend's bridal party.", "themes": ["friendship", "jealousy"], "similar": ["Girls Trip", "Pitch Perfect"]},
    {"title": "La La Land", "year": 2016, "director": "Damien Chazelle", "genres": ["Romance", "Musical"], "cast": ["Ryan Gosling", "Emma Stone"], "plot": "A jazz pianist and aspiring actress fall in love while pursuing dreams in LA.", "themes": ["love", "dreams"], "similar": ["A Star Is Born", "The Notebook"]},
    {"title": "The Notebook", "year": 2004, "director": "Nick Cassavetes", "genres": ["Romance", "Drama"], "cast": ["Ryan Gosling", "Rachel McAdams"], "plot": "A poor young man falls for a wealthy girl, but class differences threaten their love.", "themes": ["eternal love", "devotion"], "similar": ["A Walk to Remember", "Titanic"]},
    {"title": "Pride and Prejudice", "year": 2005, "director": "Joe Wright", "genres": ["Romance", "Drama"], "cast": ["Keira Knightley", "Matthew Macfadyen"], "plot": "A spirited woman and a proud gentleman overcome mutual dislike to find love.", "themes": ["love", "pride"], "similar": ["Sense and Sensibility", "Emma"]},
    {"title": "Crazy Rich Asians", "year": 2018, "director": "Jon M. Chu", "genres": ["Romance", "Comedy"], "cast": ["Constance Wu", "Henry Golding"], "plot": "A woman discovers her boyfriend is from one of Singapore's wealthiest families.", "themes": ["family", "cultural identity"], "similar": ["The Proposal", "Always Be My Maybe"]},
    {"title": "Blade Runner 2049", "year": 2017, "director": "Denis Villeneuve", "genres": ["Sci-Fi", "Drama"], "cast": ["Ryan Gosling", "Harrison Ford"], "plot": "A blade runner uncovers a secret that leads to a man missing for thirty years.", "themes": ["identity", "humanity"], "similar": ["Blade Runner", "Ex Machina"]},
    {"title": "Dune", "year": 2021, "director": "Denis Villeneuve", "genres": ["Sci-Fi", "Adventure"], "cast": ["Timothée Chalamet", "Zendaya"], "plot": "A noble family navigates politics on a desert planet holding the universe's most valuable resource.", "themes": ["destiny", "power"], "similar": ["Dune: Part Two", "Star Wars"]},
    {"title": "Arrival", "year": 2016, "director": "Denis Villeneuve", "genres": ["Sci-Fi", "Drama"], "cast": ["Amy Adams", "Jeremy Renner"], "plot": "A linguist communicates with aliens, leading to a profound discovery about time.", "themes": ["communication", "time"], "similar": ["Interstellar", "Contact"]},
    {"title": "Ex Machina", "year": 2014, "director": "Alex Garland", "genres": ["Sci-Fi", "Thriller"], "cast": ["Alicia Vikander", "Oscar Isaac"], "plot": "A programmer evaluates a humanoid AI in a groundbreaking experiment.", "themes": ["consciousness", "manipulation"], "similar": ["Her", "Blade Runner 2049"]},
    {"title": "The Martian", "year": 2015, "director": "Ridley Scott", "genres": ["Sci-Fi", "Adventure"], "cast": ["Matt Damon", "Jessica Chastain"], "plot": "An astronaut stranded on Mars must survive while NASA works to bring him home.", "themes": ["survival", "ingenuity"], "similar": ["Interstellar", "Gravity"]},
    {"title": "Avatar", "year": 2009, "director": "James Cameron", "genres": ["Sci-Fi", "Action"], "cast": ["Sam Worthington", "Zoe Saldana"], "plot": "A Marine is torn between following orders and protecting an alien world he calls home.", "themes": ["environmentalism", "identity"], "similar": ["Avatar: The Way of Water", "District 9"]},
    {"title": "Se7en", "year": 1995, "director": "David Fincher", "genres": ["Thriller", "Crime"], "cast": ["Brad Pitt", "Morgan Freeman"], "plot": "Detectives hunt a serial killer using the seven deadly sins as motifs.", "themes": ["sin", "justice"], "similar": ["Zodiac", "Silence of the Lambs"]},
    {"title": "Gone Girl", "year": 2014, "director": "David Fincher", "genres": ["Thriller", "Drama"], "cast": ["Ben Affleck", "Rosamund Pike"], "plot": "A man becomes the prime suspect in his wife's disappearance, but nothing is as it seems.", "themes": ["marriage", "deception"], "similar": ["The Girl on the Train", "Prisoners"]},
    {"title": "Prisoners", "year": 2013, "director": "Denis Villeneuve", "genres": ["Thriller", "Drama"], "cast": ["Hugh Jackman", "Jake Gyllenhaal"], "plot": "A desperate father takes matters into his own hands when his daughter goes missing.", "themes": ["morality", "justice"], "similar": ["Zodiac", "Gone Girl"]},
    {"title": "Shutter Island", "year": 2010, "director": "Martin Scorsese", "genres": ["Thriller", "Mystery"], "cast": ["Leonardo DiCaprio", "Mark Ruffalo"], "plot": "A U.S. Marshal investigates a psychiatric facility and uncovers a dark conspiracy.", "themes": ["trauma", "reality"], "similar": ["Inception", "Memento"]},
    {"title": "The Silence of the Lambs", "year": 1991, "director": "Jonathan Demme", "genres": ["Thriller", "Crime"], "cast": ["Jodie Foster", "Anthony Hopkins"], "plot": "An FBI trainee seeks help from an incarcerated cannibal to catch another killer.", "themes": ["evil", "manipulation"], "similar": ["Se7en", "Red Dragon"]},
    {"title": "Forrest Gump", "year": 1994, "director": "Robert Zemeckis", "genres": ["Drama", "Romance"], "cast": ["Tom Hanks", "Robin Wright"], "plot": "A man with low IQ accomplishes great things while witnessing key historical events.", "themes": ["destiny", "love"], "similar": ["The Shawshank Redemption", "Big Fish"]},
    {"title": "The Green Mile", "year": 1999, "director": "Frank Darabont", "genres": ["Drama", "Fantasy"], "cast": ["Tom Hanks", "Michael Clarke Duncan"], "plot": "A death row officer discovers an inmate has a mysterious gift and is innocent.", "themes": ["innocence", "miracles"], "similar": ["The Shawshank Redemption", "Forrest Gump"]},
    {"title": "Good Will Hunting", "year": 1997, "director": "Gus Van Sant", "genres": ["Drama", "Romance"], "cast": ["Matt Damon", "Robin Williams"], "plot": "A janitor at MIT has a gift for math but needs help overcoming his troubled past.", "themes": ["potential", "trauma"], "similar": ["A Beautiful Mind", "Dead Poets Society"]},
    {"title": "Fight Club", "year": 1999, "director": "David Fincher", "genres": ["Drama", "Thriller"], "cast": ["Brad Pitt", "Edward Norton"], "plot": "An insomniac and a soap salesman build an organization for men escaping modern life.", "themes": ["identity", "consumerism"], "similar": ["American Psycho", "Mr. Robot"]},
    {"title": "Schindler's List", "year": 1993, "director": "Steven Spielberg", "genres": ["Drama", "History"], "cast": ["Liam Neeson", "Ralph Fiennes"], "plot": "A businessman saves over a thousand Jewish refugees during the Holocaust.", "themes": ["humanity", "redemption"], "similar": ["The Pianist", "Life Is Beautiful"]},
    {"title": "Spider-Man: Into the Spider-Verse", "year": 2018, "director": "Various", "genres": ["Animation", "Action"], "cast": ["Shameik Moore", "Jake Johnson"], "plot": "Miles Morales becomes Spider-Man and teams up with Spider-People from other dimensions.", "themes": ["identity", "heroism"], "similar": ["Across the Spider-Verse", "The Incredibles"]},
    {"title": "Joker", "year": 2019, "director": "Todd Phillips", "genres": ["Drama", "Crime"], "cast": ["Joaquin Phoenix", "Robert De Niro"], "plot": "A failed comedian's descent into madness transforms him into the Joker.", "themes": ["mental illness", "society"], "similar": ["The Dark Knight", "Taxi Driver"]},
    {"title": "Logan", "year": 2017, "director": "James Mangold", "genres": ["Action", "Drama"], "cast": ["Hugh Jackman", "Patrick Stewart"], "plot": "A weary Logan protects a young mutant girl from dark forces in a dark future.", "themes": ["mortality", "legacy"], "similar": ["The Dark Knight", "Children of Men"]},
    {"title": "The Avengers", "year": 2012, "director": "Joss Whedon", "genres": ["Action", "Adventure"], "cast": ["Robert Downey Jr.", "Chris Evans"], "plot": "Earth's mightiest heroes unite to stop Loki and his alien army.", "themes": ["teamwork", "heroism"], "similar": ["Avengers: Endgame", "Justice League"]},
    {"title": "Black Panther", "year": 2018, "director": "Ryan Coogler", "genres": ["Action", "Adventure"], "cast": ["Chadwick Boseman", "Michael B. Jordan"], "plot": "T'Challa returns to Wakanda as King but faces a challenger threatening everything.", "themes": ["heritage", "responsibility"], "similar": ["Thor: Ragnarok", "Creed"]},
    {"title": "Coco", "year": 2017, "director": "Lee Unkrich", "genres": ["Animation", "Adventure"], "cast": ["Anthony Gonzalez", "Gael García Bernal"], "plot": "A boy enters the Land of the Dead to find his great-great-grandfather.", "themes": ["family", "memory"], "similar": ["Soul", "Encanto"]},
    {"title": "Up", "year": 2009, "director": "Pete Docter", "genres": ["Animation", "Adventure"], "cast": ["Ed Asner", "Jordan Nagai"], "plot": "An elderly widower flies his house to South America with balloons.", "themes": ["adventure", "grief"], "similar": ["Wall-E", "Inside Out"]},
    {"title": "Inside Out", "year": 2015, "director": "Pete Docter", "genres": ["Animation", "Comedy"], "cast": ["Amy Poehler", "Phyllis Smith"], "plot": "Inside a girl's mind, five emotions navigate a difficult life transition.", "themes": ["emotions", "growing up"], "similar": ["Soul", "Coco"]},
    {"title": "Toy Story", "year": 1995, "director": "John Lasseter", "genres": ["Animation", "Adventure"], "cast": ["Tom Hanks", "Tim Allen"], "plot": "A cowboy doll is threatened when a new spaceman supplants him as top toy.", "themes": ["friendship", "jealousy"], "similar": ["Toy Story 2", "Monsters, Inc."]},
    {"title": "Finding Nemo", "year": 2003, "director": "Andrew Stanton", "genres": ["Animation", "Adventure"], "cast": ["Albert Brooks", "Ellen DeGeneres"], "plot": "An overprotective clownfish searches for his captured son.", "themes": ["parenthood", "courage"], "similar": ["Finding Dory", "The Incredibles"]},
    {"title": "Amélie", "year": 2001, "director": "Jean-Pierre Jeunet", "genres": ["Comedy", "Romance"], "cast": ["Audrey Tautou", "Mathieu Kassovitz"], "plot": "A shy Parisian waitress changes lives around her while struggling with isolation.", "themes": ["kindness", "love"], "similar": ["The Grand Budapest Hotel", "Midnight in Paris"]},
    {"title": "Oldboy", "year": 2003, "director": "Park Chan-wook", "genres": ["Action", "Drama"], "cast": ["Choi Min-sik", "Yoo Ji-tae"], "plot": "After 15 years imprisoned, a man seeks vengeance on his captor.", "themes": ["revenge", "mystery"], "similar": ["I Saw the Devil", "The Handmaiden"]},
    {"title": "Your Name", "year": 2016, "director": "Makoto Shinkai", "genres": ["Animation", "Romance"], "cast": ["Ryunosuke Kamiki", "Mone Kamishiraishi"], "plot": "Two teenagers mysteriously swap bodies and form a connection across time.", "themes": ["connection", "fate"], "similar": ["Weathering with You", "A Silent Voice"]},
    {"title": "City of God", "year": 2002, "director": "Fernando Meirelles", "genres": ["Crime", "Drama"], "cast": ["Alexandre Rodrigues", "Leandro Firmino"], "plot": "Two boys in Rio take different paths: one becomes a photographer, one a drug dealer.", "themes": ["poverty", "choices"], "similar": ["Slumdog Millionaire", "Trainspotting"]},
    {"title": "Pan's Labyrinth", "year": 2006, "director": "Guillermo del Toro", "genres": ["Drama", "Fantasy"], "cast": ["Ivana Baquero", "Doug Jones"], "plot": "In post-Civil War Spain, a girl discovers a labyrinth and meets a faun.", "themes": ["innocence", "fantasy vs reality"], "similar": ["The Shape of Water", "Crimson Peak"]},
    {"title": "Oppenheimer", "year": 2023, "director": "Christopher Nolan", "genres": ["Drama", "Biography"], "cast": ["Cillian Murphy", "Robert Downey Jr."], "plot": "The story of physicist J. Robert Oppenheimer and the atomic bomb.", "themes": ["responsibility", "morality"], "similar": ["A Beautiful Mind", "The Imitation Game"]},
    {"title": "Barbie", "year": 2023, "director": "Greta Gerwig", "genres": ["Comedy", "Fantasy"], "cast": ["Margot Robbie", "Ryan Gosling"], "plot": "Barbie and Ken leave Barbieland to discover themselves in the real world.", "themes": ["identity", "feminism"], "similar": ["Legally Blonde", "The Truman Show"]},
    {"title": "The Batman", "year": 2022, "director": "Matt Reeves", "genres": ["Action", "Crime"], "cast": ["Robert Pattinson", "Zoë Kravitz"], "plot": "Batman ventures into Gotham's underworld when a killer leaves cryptic clues.", "themes": ["vengeance", "corruption"], "similar": ["The Dark Knight", "Se7en"]},
]

# Generate training examples
training_data = []

for movie in MOVIES:
    cast_str = ", ".join(movie["cast"])
    genres_str = "/".join(movie["genres"])
    themes_str = ", ".join(movie["themes"])
    similar_str = ", ".join(movie["similar"])

    # About questions
    training_data.append({"messages": [
        {"role": "system", "content": "You are CineBot, a friendly movie expert."},
        {"role": "user", "content": f"What is {movie['title']} about?"},
        {"role": "assistant", "content": f"{movie['title']} ({movie['year']}) is directed by {movie['director']}. {movie['plot']} It stars {cast_str} and explores themes of {themes_str}."}
    ]})

    # Recommendation questions
    training_data.append({"messages": [
        {"role": "system", "content": "You are CineBot, a friendly movie expert."},
        {"role": "user", "content": f"I liked {movie['title']}, what else should I watch?"},
        {"role": "assistant", "content": f"If you enjoyed {movie['title']}, I'd recommend {similar_str}! They share similar themes of {themes_str}."}
    ]})

    # Genre questions
    for genre in movie["genres"]:
        training_data.append({"messages": [
            {"role": "system", "content": "You are CineBot, a friendly movie expert."},
            {"role": "user", "content": f"Recommend a {genre} movie"},
            {"role": "assistant", "content": f"Check out {movie['title']} ({movie['year']})! {movie['plot']} It's an excellent {genre} film by {movie['director']}."}
        ]})

# General conversations
general = [
    ("Who are you?", "I'm CineBot, your movie expert! I can recommend films, answer questions about movies, and help you decide what to watch."),
    ("What can you do?", "I help with movie recommendations, answer questions about films and actors, and suggest what to watch based on your mood!"),
    ("I don't know what to watch", "Tell me your mood! Exciting action, thought-provoking drama, or maybe a fun comedy? I'll find the perfect film."),
    ("What's the best movie ever made?", "That's subjective, but The Shawshank Redemption, The Godfather, and Citizen Kane often top critics' lists. What genre do you enjoy most?"),
    ("I want something scary", "For horror, I'd suggest Hereditary for psychological terror, A Quiet Place for suspense, or The Shining for a classic. How scared do you want to be?"),
    ("Something for date night?", "For date night, try La La Land for romance, Crazy Rich Asians for a fun rom-com, or The Grand Budapest Hotel for something quirky and charming!"),
]
for q, a in general:
    training_data.append({"messages": [
        {"role": "system", "content": "You are CineBot, a friendly movie expert."},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a}
    ]})

random.shuffle(training_data)
print(f"Generated {len(training_data)} training examples")

# Save to file
with open("movie_training.jsonl", "w") as f:
    for item in training_data:
        f.write(json.dumps(item) + "\n")

Generated 260 training examples


## 4. Load Model and Setup LoRA

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL}...")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

In [10]:
%pip uninstall -y pyarrow datasets trl
%pip install -q --upgrade --force-reinstall torch transformers datasets peft trl bitsandbytes accelerate huggingface_hub

Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1
Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
Found existing installation: trl 1.0.0
Uninstalling trl-1.0.0:
  Successfully uninstalled trl-1.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
libcuvs-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
gcsfs 2025.3.0 re

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Prepare Dataset

In [ ]:
dataset = load_dataset("json", data_files="movie_training.jsonl", split="train")

def format_chat(example):
    text = ""
    for msg in example["messages"]:
        if msg["role"] == "system": text += f"<|system|>\n{msg['content']}</s>\n"
        elif msg["role"] == "user": text += f"<|user|>\n{msg['content']}</s>\n"
        elif msg["role"] == "assistant": text += f"<|assistant|>\n{msg['content']}</s>\n"
    return {"text": text}

dataset = dataset.map(format_chat)
print(f"Dataset: {len(dataset)} examples")

## 6. Train!

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=3,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    learning_rate=2e-4, warmup_ratio=0.03,
    logging_steps=10, save_steps=50,
    fp16=True, optim="paged_adamw_8bit", lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model, train_dataset=dataset, tokenizer=tokenizer,
    args=training_args, dataset_text_field="text", max_seq_length=512,
)

print(" Starting training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f" Model saved!")

## 7. Test Model

In [ ]:
test_questions = [
    "What is Inception about?",
    "Recommend something like The Matrix",
    "I want to watch a horror movie",
    "Who are you?",
]

model.eval()
for q in test_questions:
    prompt = f"<|system|>\nYou are CineBot, a friendly movie expert.</s>\n<|user|>\n{q}</s>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|assistant|>" in response: response = response.split("<|assistant|>")[-1].strip()
    print(f"\n❓ {q}\n {response[:250]}")
    print("-" * 50)

## 8. Upload to Hugging Face

In [ ]:
repo_id = f"{HF_USERNAME}/{MODEL_NAME}"
print(f"Uploading to {repo_id}...")
model.push_to_hub(repo_id, use_temp_dir=True)
tokenizer.push_to_hub(repo_id)
print(f"\nModel uploaded!")
print(f" https://huggingface.co/{repo_id}")
print(f"\n📱 For CineFlow deployment, set:")
print(f"   LLM_MODEL={repo_id}")